# history

> Undo history management for segment editing workflows

In [ ]:
#| default_exp services.history

In [ ]:
#| export
from typing import Optional, Tuple, List, Dict, Any

## Constants

In [ ]:
#| export
DEFAULT_MAX_HISTORY_DEPTH = 50

## History Operations

In [ ]:
#| export
def push_history(
    history: List[Dict[str, Any]],  # Current history stack
    segments: List[Dict[str, Any]],  # Segments to snapshot
    focused_index: int,  # Focused index to snapshot
    max_depth: int = DEFAULT_MAX_HISTORY_DEPTH,  # Maximum history depth
) -> List[Dict[str, Any]]:  # Updated history stack
    """Push a state snapshot onto the history stack, enforcing max depth."""
    new_history = list(history)
    new_history.append({
        "segments": segments,
        "focused_index": focused_index,
    })
    if len(new_history) > max_depth:
        new_history = new_history[-max_depth:]
    return new_history

In [ ]:
#| export
def pop_history(
    history: List[Dict[str, Any]],  # Current history stack
) -> Optional[Tuple[List[Dict[str, Any]], int, List[Dict[str, Any]]]]:  # (segments, focused_index, updated_history) or None
    """Pop the most recent snapshot from the history stack, clamping the focused index."""
    if not history:
        return None
    
    new_history = list(history)
    previous_state = new_history.pop()
    segments = previous_state["segments"]
    focused_index = previous_state["focused_index"]
    
    # Clamp focus to valid range
    max_index = max(0, len(segments) - 1)
    focused_index = min(focused_index, max_index)
    
    return segments, focused_index, new_history

## Tests

In [ ]:
# Push to empty history
h = push_history([], [{"text": "a"}], 0)
assert len(h) == 1
assert h[0]["segments"] == [{"text": "a"}]
assert h[0]["focused_index"] == 0

# Push preserves existing entries
h = push_history(h, [{"text": "b"}], 1)
assert len(h) == 2
assert h[1]["focused_index"] == 1

# Max depth enforcement
h = push_history([], [{"text": "old"}], 0, max_depth=2)
h = push_history(h, [{"text": "mid"}], 0, max_depth=2)
h = push_history(h, [{"text": "new"}], 0, max_depth=2)
assert len(h) == 2
assert h[0]["segments"] == [{"text": "mid"}]
assert h[1]["segments"] == [{"text": "new"}]

# Original list is not mutated
original = [{"segments": [{"text": "x"}], "focused_index": 0}]
result = push_history(original, [{"text": "y"}], 1)
assert len(original) == 1
assert len(result) == 2

print("push_history tests passed")

push_history tests passed


In [ ]:
# Pop from empty history returns None
assert pop_history([]) is None

# Pop returns last entry and updated history
h = [
    {"segments": [{"text": "first"}], "focused_index": 0},
    {"segments": [{"text": "second"}], "focused_index": 1},
]
segments, idx, remaining = pop_history(h)
assert segments == [{"text": "second"}]
assert idx == 0  # clamped: only 1 segment, so max valid index is 0
assert len(remaining) == 1

# Focused index clamping
h = [{"segments": [{"text": "a"}, {"text": "b"}], "focused_index": 5}]
segments, idx, remaining = pop_history(h)
assert idx == 1  # clamped to max valid index
assert len(remaining) == 0

# Original list is not mutated
original = [{"segments": [{"text": "x"}], "focused_index": 0}]
_, _, _ = pop_history(original)
assert len(original) == 1

print("pop_history tests passed")

pop_history tests passed


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()